## Erstellung von $V_{eval}$ 

### Rohdaten holen

Ursprung der Daten ist ein `jsonl` von Lorenz, das ich am 7. August per Mail zugeschickt bekommen habe. Diese enthalten zwar den Rohtext der Wiki-Artikel, aber auch viel Noise, das bei der Fragengenerierung stören würde. Daher müssen wir die Datenbasis separat von Wikipedia holen.

Der Datensatz besteht aus 300 Fragne. Die JSON der `jsonl` haben alle dieselbe Struktur. Es gibt ein Feld `document_url` mit einem Link zum Wikipedia Artikel, auf dem die Frage basiert. Im URL steckt auch ein KWARG `oldid`, das eine spezifische Version des Wiki-Artikels referenziert. So kann man exakt die Datenbasis von ClapNQ herstellen.

### Cleaning

Die Rohdaten von Wikipedia sind stark geschachtelt und enthalten noch HTML-ähnlichen Syntax. Wir verwenden `hwparserfromhell` (vorschlag von ChatGPT) zum Cleanen. Danach ist der Artikel ein Flaches JSON, das nach Sections organisiert ist. Hier hab ich noch eine Blacklist definieren lassen mit Sections, die für uns keinen Informationswert haben.

`"references", "notes", "notes and references", "footnotes", "citations", "works cited", "further reading", "bibliography", "sources", "external links", "see also", "related articles", "gallery"`

Refs: https://github.com/earwig/mwparserfromhell

### Chunking

Der Korpus wird mittels rekursivem, zeichenbasiertem Chunking segmentiert: Feste Chunk-Größen (3500 Zeichen) und überlappende Bereiche (525 Zeichen). Der gesamte Artikeltext wird am Stück gesplittet. Die genaue Sektionszugehörigkeit eines Chunks ($d_i$) wird anschließend über ein Index-Mapping der Originaltexte rückwirkend ermittelt.

Jeder Chunk wird mit zwei Arten von Metadaten angereichert:
* **Deskriptive Metadaten:** Der Textkörper erhält Dokumenten-Header (Artikel- und Sektions-Titel), um den ursprünglichen Kontext zu bewahren.
* **Strukturelle Metadaten:** Für die deterministische Kontexterweiterung speichert die ChromaDB die Dokumenten-ID $P(d_i)$, die sequenzielle Position $Idx(d_i)$ sowie die Gesamtanzahl der Chunks $N(d_i)$.


In [2]:
import json

file_path = "../data/clapnq_dev_answerable_orig.jsonl"

all_keys = set()
key_structures = set()
total_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        total_lines += 1
        data = json.loads(line)
        # Speichere die Keys als sortiertes Tuple zur Überprüfung
        keys = tuple(sorted(data.keys()))
        key_structures.add(keys)

print(f"Geprüfte Zeilen: {total_lines}")

if len(key_structures) == 1:
    print("✅ Die Struktur (Keys) ist in ALLEN Zeilen exakt identisch!")
    print("\nVorhandene Schlüssel:")
    for k in list(key_structures)[0]:
        print(f" - {k}")
else:
    print(
        f"⚠️ Abweichungen gefunden! Es gibt {len(key_structures)} verschiedene Key-Kombinationen."
    )

Geprüfte Zeilen: 300
✅ Die Struktur (Keys) ist in ALLEN Zeilen exakt identisch!

Vorhandene Schlüssel:
 - annotations
 - document_plaintext
 - document_title
 - document_url
 - example_id
 - language
 - passage_answer_candidates
 - question_text


In [21]:
urls = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        urls.append(data["document_url"])
len(urls)

300

## fetch articles

oldid already refers to a specific db-entry of the Wikipedia-Article -> we're consistent with the ClapnQ information

die Wikipedia-Artikel sind noch "roh" und enthalten HTML  links etc, daher `mwparserfromhell` um zu parsen

In [24]:
import os
import json
import requests
import html
import re
import mwparserfromhell
from urllib.parse import urlparse, parse_qs

OUTPUT_DIR = "../data/wikipedia_articles"

def extract_oldid(url):
    """Extrahiert die oldid aus der URL."""
    decoded_url = html.unescape(url)
    parsed_url = urlparse(decoded_url)
    query_params = parse_qs(parsed_url.query)
    
    if 'oldid' in query_params:
        return query_params['oldid'][0]
    return None

def fetch_and_clean_article(oldid):
    """Holt den Wikitext und parst ihn mit mwparserfromhell in sauberen Text."""
    api_url = "https://en.wikipedia.org/w/api.php"
    
    # WICHTIG: User-Agent, um den 403 Forbidden Error zu vermeiden!
    headers = {
        "User-Agent": "WikipediaQABuilder/1.0 (daniel.hillebrand@uibk.ac.at) - Python Script"
    }
    
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "revids": oldid,
        "format": "json",
        "rvslots": "main"
    }

    response = requests.get(api_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()

    pages = data.get("query", {}).get("pages", {})
    if not pages or "-1" in pages:
        print(f"❌ Revision {oldid} nicht gefunden.")
        return None

    page = list(pages.values())[0]
    title = page.get("title", "Unknown_Title")
    
    try:
        wikitext = page["revisions"][0]["slots"]["main"]["*"]
    except KeyError:
        print(f"❌ Kein Wikitext für {oldid} gefunden.")
        return None

    # 1. Parse den Wikitext in einen Abstract Syntax Tree (AST)
    wikicode = mwparserfromhell.parse(wikitext)

    # 2. Entferne alle <ref> Tags und HTML-Kommentare komplett aus dem Baum
    for tag in wikicode.filter_tags():
        if tag.tag.lower() == "ref":
            try:
                wikicode.remove(tag)
            except ValueError:
                pass
                
    for comment in wikicode.filter_comments():
        try:
            wikicode.remove(comment)
        except ValueError:
            pass

    sections_data = []
    
    # 3. Teile den Artikel in Sections auf
    for section in wikicode.get_sections(include_lead=True, flat=True):
        
        # Finde heraus, ob es eine Section oder Subsection ist
        headings = section.filter_headings()
        if headings:
            heading = headings[0]
            section_title = str(heading.title).strip()
            level = heading.level  # 2 für == Section ==, 3 für === Subsection ===
            # Entferne die Überschrift aus dem Textkörper
            section.remove(heading)
        else:
            section_title = "Introduction"
            level = 1

        # 4. Magie: strip_code() entfernt Infoboxen, löst Links auf und macht Plain-Text daraus
        clean_text = section.strip_code().strip()
        
        # Überflüssige leere Zeilen bereinigen
        clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)

        # 5. Nur hinzufügen, wenn wirklich Text übrig geblieben ist!
        if clean_text:
            sections_data.append({
                "section_title": section_title,
                "level": level,
                "content": clean_text
            })

    return {
        "title": title,
        "oldid": oldid,
        "url": f"https://en.wikipedia.org/w/index.php?title={title.replace(' ', '_')}&oldid={oldid}",
        "sections": sections_data
    }

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"📁 Verzeichnis '{OUTPUT_DIR}' ist bereit.")

    # Lese alle bereits existierenden Dateien im Zielordner ein
    existing_files = os.listdir(OUTPUT_DIR)

    for url in urls:
        oldid = extract_oldid(url)
        if not oldid:
            print(f"⚠️ Keine oldid in {url} gefunden. Überspringe.")
            continue
            
        # LOGIK HINZUGEFÜGT: Überspringen, falls eine Datei mit dieser oldid bereits existiert
        if any(f.endswith(f"_{oldid}.json") for f in existing_files):
            print(f"⏭️ Datei für oldid {oldid} existiert bereits. Überspringe.")
            continue
            
        print(f"⏳ Hole und säubere oldid: {oldid}...")
        structured_data = fetch_and_clean_article(oldid)
        
        if structured_data:
            safe_title = structured_data["title"].replace(" ", "_").replace("/", "-")
            filename = f"{safe_title}_{oldid}.json"
            filepath = os.path.join(OUTPUT_DIR, filename)
            
            with open(filepath, "w", encoding="utf-8") as f:
                json.dump(structured_data, f, indent=4, ensure_ascii=False)
                
            print(f"✅ Gespeichert: {filepath}")
            
            # Füge die neu erstellte Datei zur Liste hinzu (verhindert doppelte Downloads, falls die URL mehrfach in der Liste ist)
            existing_files.append(filename)

if __name__ == "__main__":
    main()

📁 Verzeichnis '../data/wikipedia_articles' ist bereit.
⏭️ Datei für oldid 853705998 existiert bereits. Überspringe.
⏭️ Datei für oldid 812042411 existiert bereits. Überspringe.
⏭️ Datei für oldid 843082151 existiert bereits. Überspringe.
⏭️ Datei für oldid 811525318 existiert bereits. Überspringe.
⏭️ Datei für oldid 815597210 existiert bereits. Überspringe.
⏭️ Datei für oldid 852859365 existiert bereits. Überspringe.
⏭️ Datei für oldid 854238370 existiert bereits. Überspringe.
⏭️ Datei für oldid 806130864 existiert bereits. Überspringe.
⏭️ Datei für oldid 830825344 existiert bereits. Überspringe.
⏭️ Datei für oldid 836192773 existiert bereits. Überspringe.
⏭️ Datei für oldid 844481105 existiert bereits. Überspringe.
⏭️ Datei für oldid 843757803 existiert bereits. Überspringe.
⏭️ Datei für oldid 849504314 existiert bereits. Überspringe.
⏭️ Datei für oldid 842789562 existiert bereits. Überspringe.
⏭️ Datei für oldid 821698509 existiert bereits. Überspringe.
⏭️ Datei für oldid 833898472 e

für wikitext mit id 857007246 wurde kein text gefunden -> also nur mehr 299

In [25]:
import os
import json

# Pfade definieren
INPUT_DIR = "../data/wikipedia_articles"
OUTPUT_DIR = "../data/wikipedia_articles_cleaned"

def is_junk_section(title):
    """Prüft, ob eine Section aussortiert werden soll."""
    # Alles in Kleinschreibung umwandeln und Leerzeichen am Rand entfernen
    clean_title = title.lower().strip()
    
    # Bekannte Exakt-Matches (Standard Wikipedia Anhänge)
    junk_exact = {
        "references", "notes", "notes and references", "footnotes", 
        "citations", "works cited", "further reading", "bibliography", 
        "sources", "external links", "see also", "related articles",
        "gallery"
    }
    
    if clean_title in junk_exact:
        return True
        
    # Substring-Matches für unsaubere Benennungen
    if "external link" in clean_title or "further reading" in clean_title:
        return True
        
    return False

def main():
    # 1. Prüfen, ob der Input-Ordner existiert
    if not os.path.exists(INPUT_DIR):
        print(f"❌ Fehler: Der Ordner '{INPUT_DIR}' existiert nicht.")
        return

    # 2. Output-Ordner erstellen (falls noch nicht vorhanden)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"📁 Lese aus '{INPUT_DIR}' ...")
    print(f"📁 Speichere in '{OUTPUT_DIR}' ...\n")

    processed_count = 0

    # 3. Alle JSON-Dateien im Ordner durchgehen
    for filename in os.listdir(INPUT_DIR):
        if not filename.endswith(".json"):
            continue

        input_path = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, filename)

        # Datei einlesen
        with open(input_path, "r", encoding="utf-8") as f:
            try:
                article_data = json.load(f)
            except json.JSONDecodeError:
                print(f"⚠️ Fehler beim Lesen von {filename} (Kein valides JSON). Überspringe.")
                continue

        original_sections = article_data.get("sections", [])
        cleaned_sections = []
        removed_titles = []

        # Sections filtern
        for sec in original_sections:
            title = sec.get("section_title", "")
            if not is_junk_section(title):
                cleaned_sections.append(sec)
            else:
                removed_titles.append(title)

        # Die bereinigte Liste wieder ins Dictionary schreiben
        article_data["sections"] = cleaned_sections

        # Die saubere Version im neuen Ordner speichern
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(article_data, f, indent=4, ensure_ascii=False)

        processed_count += 1
        
        # Log-Ausgabe für Transparenz
        removed_str = ", ".join(removed_titles) if removed_titles else "Keine"
        print(f"✅ {filename}")
        print(f"   Sections behalten: {len(cleaned_sections)} / {len(original_sections)}")
        print(f"   Entfernt: [{removed_str}]\n")

    print(f"🎉 Fertig! {processed_count} Dateien wurden bereinigt und gespeichert.")

if __name__ == "__main__":
    main()

📁 Lese aus '../data/wikipedia_articles' ...
📁 Speichere in '../data/wikipedia_articles_cleaned' ...

✅ Kingdom_of_England_864500956.json
   Sections behalten: 12 / 14
   Entfernt: [See also, Bibliography]

✅ United_States_embargo_against_Cuba_800332980.json
   Sections behalten: 17 / 19
   Entfernt: [See also, External links]

✅ Berlin_Wall_843537822.json
   Sections behalten: 31 / 34
   Entfernt: [See also, References, External links]

✅ All-way_stop_836778170.json
   Sections behalten: 5 / 7
   Entfernt: [See also, References]

✅ Dodo_834509188.json
   Sections behalten: 17 / 20
   Entfernt: [See also, Sources, External links]

✅ Moai_826099141.json
   Sections behalten: 16 / 19
   Entfernt: [See also, References, External links]

✅ Jessica_Jones_842402029.json
   Sections behalten: 11 / 12
   Entfernt: [External links]

✅ Impact_of_geography_on_colonial_America_837238085.json
   Sections behalten: 16 / 17
   Entfernt: [References]

✅ Governor_General_of_Canada_804155181.json
   Sect

jetzt müssen wir die dokumente parsen mit overlap, ca. 1000 tokens = chunk_size 3500

In [1]:
import os
import shutil
import json
import time
import chromadb
from google import genai
from google.genai import types
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# --- 1. Konfiguration ---
load_dotenv(override=True)
client = genai.Client()

# Dein angepasster Pfad
INPUT_DIR = "/home/qe/git_projects/masterarbeit/data/wikipedia_articles_cleaned"
DB_PATH = "./chroma_db_wiki"

if os.path.exists(DB_PATH):
    print(f"🗑️ Lösche alte ChromaDB-Daten in {DB_PATH}...")
    try:
        shutil.rmtree(DB_PATH)
        print("✅ Alte Daten gelöscht.")
    except PermissionError:
        print("⚠️ HINWEIS: Datei ist blockiert! Bitte starte den Jupyter Kernel neu.")

# --- 3. ChromaDB sauber initialisieren ---
print("⚙️ Initialisiere frische ChromaDB...")
chroma_client = chromadb.PersistentClient(path=DB_PATH)
vector_collection = chroma_client.get_or_create_collection(name="wikipedia_eval_chunks")

# --- 4. Text Splitter & Pipeline ---
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    chunk_size=3500,
    chunk_overlap=525,
    length_function=len,
    is_separator_regex=True
)

def get_embeddings_with_retry(texts):
    SUB_BATCH_SIZE = 100 
    all_embeddings = []

    for i in range(0, len(texts), SUB_BATCH_SIZE):
        batch = texts[i:i + SUB_BATCH_SIZE]
        while True:
            try:
                response = client.models.embed_content(
                    model='gemini-embedding-001',
                    contents=batch,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                all_embeddings.extend([emb.values for emb in response.embeddings])
                break
            except Exception as e:
                if "429" in str(e):
                    print("⚠️ Rate Limit erreicht. Warte 5 Sekunden...")
                    time.sleep(5) 
                else:
                    raise e
    return all_embeddings

def process_everything():
    if not os.path.exists(INPUT_DIR):
        print(f"❌ Input-Verzeichnis {INPUT_DIR} nicht gefunden.")
        return

    json_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json')]
    print(f"🚀 Starte Ingestion von {len(json_files)} bereinigten Wikipedia-Artikeln...")
    
    pbar = tqdm(total=len(json_files), desc="Artikel verarbeitet")

    for filename in json_files:
        filepath = os.path.join(INPUT_DIR, filename)
        
        with open(filepath, "r", encoding="utf-8") as f:
            article = json.load(f)
            
        article_title = article.get("title", "Unknown")
        parent_id = str(article.get("oldid", filename)) 
        
        # --- LOGIK: Ganzes Dokument am Stück mit Positions-Mapping ---
        full_text = ""
        section_map = [] # Speichert: (start_index, end_index, section_title)
        
        # 1. Text zusammenbauen und Sektions-Grenzen merken
        for section in article.get("sections", []):
            sec_title = section.get("section_title", "Unknown")
            content = section.get("content", "")
            
            if not content:
                continue
                
            start_idx = len(full_text)
            full_text += content + "\n\n"
            end_idx = len(full_text)
            
            section_map.append((start_idx, end_idx, sec_title))
            
        # 2. Den kompletten Text am Stück mit echtem Overlap splitten
        raw_chunks = text_splitter.split_text(full_text)
        
        article_chunks_data = []
        search_pos = 0 # Hilfsvariable, um den Chunk im Originaltext zu finden
        
        for chunk_text in raw_chunks:
            # Finde die Position des Chunks im Gesamttext
            chunk_start_idx = full_text.find(chunk_text, search_pos)
            if chunk_start_idx == -1: 
                chunk_start_idx = search_pos # Fallback, sollte nie passieren
                
            chunk_end_idx = chunk_start_idx + len(chunk_text)
            search_pos = chunk_start_idx + 1 # Für die nächste Suche weiterrücken
            
            # Prüfe, welche Sektionen sich mit diesem Chunk überschneiden
            overlapping_sections = []
            for s_start, s_end, s_title in section_map:
                # Prüfe auf Überschneidung der Intervalle
                if max(chunk_start_idx, s_start) < min(chunk_end_idx, s_end):
                    if s_title not in overlapping_sections:
                        overlapping_sections.append(s_title)
            
            # Sektionen zusammenfügen (falls es überlappt, z.B. "Intro, Career")
            sec_title_str = ", ".join(overlapping_sections) if overlapping_sections else "Unknown"
            
            # Rückwirkende Anreicherung
            enriched_text = f"Title: {article_title}\nSection(s): {sec_title_str}\n\n{chunk_text}"
            article_chunks_data.append(enriched_text)
            
        total_chunks = len(article_chunks_data)
        # ------------------------------------------------------------------
        
        if total_chunks == 0:
            pbar.update(1)
            continue
            
        all_chunk_ids = []
        all_chunk_metas = []
        
        for i, enriched_text in enumerate(article_chunks_data):
            all_chunk_ids.append(f"{parent_id}#chunk_{i}")
            
            all_chunk_metas.append({
                "parent_id": parent_id,            
                "chunk_index": i,                  
                "total_chunks": total_chunks,      
                "article_title": article_title
            })

        try:
            embeddings = get_embeddings_with_retry(article_chunks_data)

            vector_collection.add(
                ids=all_chunk_ids,
                embeddings=embeddings,
                documents=article_chunks_data,
                metadatas=all_chunk_metas
            )
            time.sleep(0.1) 
        except Exception as e:
            print(f"\n❌ Fehler beim Verarbeiten von {filename}: {e}")
            
        pbar.update(1)
        
    pbar.close()
    print("✅ Ingestion abgeschlossen!")

if __name__ == "__main__":
    process_everything()

🗑️ Lösche alte ChromaDB-Daten in ./chroma_db_wiki...
✅ Alte Daten gelöscht.
⚙️ Initialisiere frische ChromaDB...
🚀 Starte Ingestion von 299 bereinigten Wikipedia-Artikeln...


Artikel verarbeitet:  52%|█████████▊         | 154/299 [01:38<01:11,  2.03it/s]


❌ Fehler beim Verarbeiten von Context_effect_840750606.json: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}


Artikel verarbeitet: 100%|███████████████████| 299/299 [03:04<00:00,  1.62it/s]

✅ Ingestion abgeschlossen!


In [2]:
# patch für die eine fehlende datei

import os
import json
import time
import chromadb
from google import genai
from google.genai import types
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)
client = genai.Client()

INPUT_DIR = "/home/qe/git_projects/masterarbeit/data/wikipedia_articles_cleaned"
DB_PATH = "./chroma_db_wiki"
MISSING_FILE = "Context_effect_840750606.json" # Der fehlende Artikel

print("⚙️ Verbinde mit bestehender ChromaDB...")
chroma_client = chromadb.PersistentClient(path=DB_PATH)
vector_collection = chroma_client.get_collection(name="wikipedia_eval_chunks")

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    chunk_size=3500, chunk_overlap=525, length_function=len, is_separator_regex=True
)

filepath = os.path.join(INPUT_DIR, MISSING_FILE)

with open(filepath, "r", encoding="utf-8") as f:
    article = json.load(f)

article_title = article.get("title", "Unknown")
parent_id = str(article.get("oldid", MISSING_FILE)) 

full_text = ""
section_map = [] 

for section in article.get("sections", []):
    sec_title = section.get("section_title", "Unknown")
    content = section.get("content", "")
    if not content: continue
        
    start_idx = len(full_text)
    full_text += content + "\n\n"
    end_idx = len(full_text)
    section_map.append((start_idx, end_idx, sec_title))

raw_chunks = text_splitter.split_text(full_text)
article_chunks_data = []
search_pos = 0 

for chunk_text in raw_chunks:
    chunk_start_idx = full_text.find(chunk_text, search_pos)
    if chunk_start_idx == -1: chunk_start_idx = search_pos 
    chunk_end_idx = chunk_start_idx + len(chunk_text)
    search_pos = chunk_start_idx + 1 
    
    overlapping_sections = []
    for s_start, s_end, s_title in section_map:
        if max(chunk_start_idx, s_start) < min(chunk_end_idx, s_end):
            if s_title not in overlapping_sections:
                overlapping_sections.append(s_title)
    
    sec_title_str = ", ".join(overlapping_sections) if overlapping_sections else "Unknown"
    enriched_text = f"Title: {article_title}\nSection(s): {sec_title_str}\n\n{chunk_text}"
    article_chunks_data.append(enriched_text)

total_chunks = len(article_chunks_data)
all_chunk_ids = []
all_chunk_metas = []

for i, enriched_text in enumerate(article_chunks_data):
    all_chunk_ids.append(f"{parent_id}#chunk_{i}")
    all_chunk_metas.append({
        "parent_id": parent_id,            
        "chunk_index": i,                  
        "total_chunks": total_chunks,      
        "article_title": article_title
    })

print(f"🔄 Sende {total_chunks} Chunks an Gemini API...")
all_embeddings = []
for i in range(0, len(article_chunks_data), 100):
    batch = article_chunks_data[i:i + 100]
    response = client.models.embed_content(
        model='gemini-embedding-001', contents=batch,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
    )
    all_embeddings.extend([emb.values for emb in response.embeddings])

vector_collection.add(
    ids=all_chunk_ids,
    embeddings=all_embeddings,
    documents=article_chunks_data,
    metadatas=all_chunk_metas
)

print(f"✅ Artikel {MISSING_FILE} erfolgreich nachträglich importiert!")

⚙️ Verbinde mit bestehender ChromaDB...
🔄 Sende 3 Chunks an Gemini API...
✅ Artikel Context_effect_840750606.json erfolgreich nachträglich importiert!


In [2]:
import chromadb

DB_PATH = "./chroma_db_wiki"

# Verbindung zur ChromaDB herstellen
chroma_client = chromadb.PersistentClient(path=DB_PATH)
vector_collection = chroma_client.get_or_create_collection(
    name="wikipedia_eval_chunks"
)

# Anzahl aller gespeicherten Chunks abfragen
total_chunks = vector_collection.count()

print(f"📊 Anzahl aller Chunks in v_eval: {total_chunks}")

📊 Anzahl aller Chunks in v_eval: 4573


In [3]:
import random
import chromadb

DB_PATH = "./chroma_db_wiki"

# Verbindung zur ChromaDB herstellen
chroma_client = chromadb.PersistentClient(path=DB_PATH)
vector_collection = chroma_client.get_or_create_collection(
    name="wikipedia_eval_chunks"
)

# Alle IDs aus der Datenbank abfragen
all_ids = vector_collection.get()["ids"]

if len(all_ids) == 0:
    print("⚠️ Die Datenbank enthält noch keine Chunks!")
else:
    # Zufällig 3 IDs auswählen (oder weniger, falls die DB kleiner ist)
    sample_size = min(3, len(all_ids))
    random_ids = random.sample(all_ids, sample_size)

    # Die Daten für diese zufälligen IDs abrufen
    samples = vector_collection.get(
        ids=random_ids, include=["documents", "metadatas"]
    )

    print(f"🎲 Zufällige Strichprobe ({sample_size} Chunks):\n" + "=" * 60)

    for i in range(sample_size):
        chunk_id = samples["ids"][i]
        metadata = samples["metadatas"][i]
        content = samples["documents"][i]

        print(f"\n📌 CHUNK #{i+1} [ID: {chunk_id}]")
        print(f"   • Artikel: {metadata.get('article_title')}")
        print(f"   • Parent ID P(d_i): {metadata.get('parent_id')}")
        print(
            f"   • Position Idx(d_i): Chunk {metadata.get('chunk_index')} von {metadata.get('total_chunks')}"
        )
        print("-" * 60)
        print("Inhalt:")
        print(content)
        print("=" * 60)

🎲 Zufällige Strichprobe (3 Chunks):

📌 CHUNK #1 [ID: 812042411#chunk_3]
   • Artikel: University of Alabama traditions
   • Parent ID P(d_i): 812042411
   • Position Idx(d_i): Chunk 3 von 4
------------------------------------------------------------
Inhalt:
Title: University of Alabama traditions
Section(s): Rammer Jammer Cheer

The cadence of the cheer was adapted from the Ole Miss cheer "Hotty Toddy" after then Ole Miss marching band director Dr. James Ferguson was appointed director of the Million Dollar Band. The cheer was long referred to as "Ole Miss", and today the drum major's signal is still the motioning of one arm in a full circle (an 'O').

The cheer was a pregame ritual until the early 2000s, chanting "We're gonna' beat the hell out of you!", but this was considered unsportsmanlike and banned. The university also briefly forbade the Million Dollar Band from playing it after games, because of its taunting nature. The move was met with a significant amount of criticism. In 